## Overview - Company Fundamentals

Extracts company fundamental data from the Alpha Vantage `OVERVIEW` endpoint.
This includes sector, market cap, P/E ratio, EPS, dividend yield, and other static metadata about each ticker.

Two functions are defined here and intended to be called by an orchestrator notebook:
- `overview_bronze(symbol, api_key)` - calls the API and returns the raw JSON payload
- `overview_silver(payload)` - transforms the raw payload into a single-row DataFrame

> **Note:** Unlike OHLCV, the `OVERVIEW` endpoint returns a flat JSON object (one record per symbol), not a time series.
> This is why `overview_silver` produces a single-row DataFrame rather than one row per date.

> **Rate limit:** Alpha Vantage free tier allows 25 requests/day. Overview calls are recommended on a weekly schedule to preserve quota.

### Setup

Load dependencies and read the API key from the `.env` file.
Make sure you have a `.env` file at the project root with `ALPHA_VANTAGE_API_KEY=your_key_here`.

In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
api_key = os.environ["ALPHA_VANTAGE_API_KEY"]

### Constants

In [ ]:
base_url = "https://www.alphavantage.co/query"

### Extraction Functions

The two functions below implement the **bronze/silver pattern**:
- **Bronze** stores the raw, unmodified API response.
- **Silver** transforms the flat JSON into a single-row DataFrame.

Both functions receive `(symbol, api_key)` in the same order as the OHLCV notebook, keeping the interface consistent for the orchestrator.

#### Bronze

Calls the `OVERVIEW` endpoint and returns the raw JSON as a Python dict.
The API returns an empty dict `{}` for unknown symbols or when quota is exceeded.

In [ ]:
def overview_bronze(symbol: str, api_key: str) -> dict:
    params = {"apikey": api_key, "function": "OVERVIEW", "symbol": symbol}
    response = requests.get(base_url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()

    if not payload.get("Symbol"):
        raise ValueError(f"Unexpected answer for {symbol}: {payload}")

    return payload

#### Silver

Receives the raw payload dict (from `overview_bronze`) and returns a single-row DataFrame.
The `OVERVIEW` response is a flat key-value object, so `pd.DataFrame(payload, index=[0])` creates one row with all fields as columns.

In [ ]:
def overview_silver(payload: dict) -> pd.DataFrame:
    return pd.DataFrame(payload, index=[0])